**The Deutsch-Jozsa algorithm.**
Deutsch-Jozsa generalizes Deutsch's algorithm to $n$-bit input functions
$f:\{0,1\}^n \to \{0,1\}$.
The problem: given a black-box oracle for $f$, determine whether $f$ is
**constant** (same output for all inputs) or **balanced** (exactly half
the inputs give 0, half give 1).
A classical randomized algorithm needs $O(2^{n-1}+1)$ queries in the worst case;
Deutsch-Jozsa solves it with **one** quantum query.

This notebook implements $n=2$ (two query qubits plus one ancilla, 8-dimensional
state space).
The circuit structure is:

1. Initialize query register $|00\rangle$, ancilla $|1\rangle$
2. Apply $H^{\otimes 2}$ to query, $H$ to ancilla
3. Apply oracle $U_f$
4. Apply $H^{\otimes 2}$ to query only
5. Inspect the final `state vector`:
   - If all indices 2-7 are **zero**, the query
   register is entirely in $\lvert 00\rangle$ $\Rightarrow$<span style="color: green;"> **constant**</span>
   - If any of indices 2-7 are **nonzero**, amplitude has spread to other query
   states $\Rightarrow$<span style="color: green;"> **balanced**</span>
   - This is equivalent to measuring the query qubits and getting
   all zeros (constant) or any nonzero result (balanced)
   - The code uses direct statevector inspection instead of qubit measurement.

---
The state vector is ordered $\lvert q_1\,q_0\,\text{anc}\rangle$,
giving 8 elements indexed 0-7:

| Index | State | Query | Ancilla |
|---|---|---|---|
| 0 | $\lvert 000\rangle$ | <span style="color:red;">**$\lvert 00\rangle$**</span> | $\lvert 0\rangle$ |
| 1 | $\lvert 001\rangle$ | <span style="color:red;">**$\lvert 00\rangle$**</span> | $\lvert 1\rangle$ |
| 2 | $\lvert 010\rangle$ | $\lvert 01\rangle$ | $\lvert 0\rangle$ |
| 3 | $\lvert 011\rangle$ | $\lvert 01\rangle$ | $\lvert 1\rangle$ |
| 4 | $\lvert 100\rangle$ | $\lvert 10\rangle$ | $\lvert 0\rangle$ |
| 5 | $\lvert 101\rangle$ | $\lvert 10\rangle$ | $\lvert 1\rangle$ |
| 6 | $\lvert 110\rangle$ | $\lvert 11\rangle$ | $\lvert 0\rangle$ |
| 7 | $\lvert 111\rangle$ | $\lvert 11\rangle$ | $\lvert 1\rangle$ |

Indices 0-1 are the only states where the query register is $\lvert 00\rangle$
(ancilla free to be either $\lvert 0\rangle$ or $\lvert 1\rangle$).
Indices 2-7 are all states where the query register is anything other than $\lvert 00\rangle$.

After the final $H^{\otimes 2}$ on the query register, Deutsch-Jozsa guarantees:
- **Constant** $f$: the query collapses to $\lvert 00\rangle$ with certainty,
  so indices <span style="color: green;">**2-7 are exactly zero**</span>
   which is precisely what `np.isclose(g4[2:], 0).all()` tests.
- **Balanced** $f$: the query has zero probability of being $\lvert 00\rangle$,
  so indices <span style="color: green;">**0-1 are exactly zero**</span> and all amplitude lives in indices 2-7.

---
**Cell 01 - Setup and Deutsch-Jozsa function.**
The function `deutsch_jozsa(u_f)` takes an $8\times 8$ unitary oracle matrix
and runs the full algorithm:
it applies $H^{\otimes 2}$ to the two-qubit query register,
$H$ to the ancilla, then the oracle $U_f$, then $H^{\otimes 2}$ to the
query register again, and finally checks whether elements 2-7 of
the output state are all zero.
The first oracle `u_f1` is balanced: inputs $00$ and $01$ flip the ancilla
($f=1$) while inputs $10$ and $11$ do not ($f=0$) - exactly half each.

In [ ]:
"""deutsch_jozsa.ipynb"""

# Cell 01 - Setup and Deutsch-Jozsa algorithm (n=2 query qubits + 1 ancilla)
# State space: 8-dimensional, ordered |q1 q0 anc>
# Oracle U_f: |x>|y> -> |x>|y XOR f(x)>
# Constant f: output concentrates on |00 anc> (elements 0-1)
# Balanced f: elements 2-7 are nonzero

import numpy as np
from IPython.display import display

from qis101_utils import as_latex

q_0 = np.array([1, 0], dtype=complex)  # |0>
q_1 = np.array([0, 1], dtype=complex)  # |1>

# Identity gate for ancilla
g_I = np.eye(2, dtype=complex)
# Hadamard gate for single qubit
g_H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
# H x H for two-qubit query register
g_H2 = np.kron(g_H, g_H)


def deutsch_jozsa(u_f: np.ndarray) -> None:
    """Run the Deutsch-Jozsa algorithm for a given 8x8 oracle matrix.

    Parameters
    ----------
    u_f : ndarray, shape (8, 8)
        Unitary oracle matrix encoding f:{0,1}^2 -> {0,1}.
        Acts as |x>|y> -> |x>|y XOR f(x)>.
    """
    x = np.kron(q_0, q_0)  # query register: |q1=0, q0=0>
    y = q_1.copy()  # ancilla: |1>

    g1 = g_H2 @ x  # H x H applied to query: |++>
    g2 = g_H @ y  # H applied to ancilla:   |->

    t1 = np.kron(g1, g2)  # joint state: |++> x |->
    g3 = u_f @ t1  # oracle applied

    t2 = np.kron(g_H2, g_I)  # H x H x I: act on query only
    g4 = t2 @ g3  # final Hadamards on query register

    # Elements 0-1 = query |00>, elements 2-7 = query |01>,|10>,|11>
    # Constant f: all amplitude in elements 0-1 -> elements 2-7 are zero
    # Balanced f: elements 2-7 are nonzero
    if np.isclose(g4[2:], 0).all():
        print("Constant")
    else:
        print("Balanced")


# u_f1: Balanced - f(00)=1, f(01)=1, f(10)=0, f(11)=0
# Rows 0-3: swap pairs (0<->1, 2<->3) = flip ancilla for inputs 00, 01
# Rows 4-7: identity = no flip for inputs 10, 11
u_f1 = np.zeros((8, 8), dtype=complex)
u_f1[0, 1] = u_f1[1, 0] = u_f1[2, 3] = u_f1[3, 2] = 1
u_f1[4, 4] = u_f1[5, 5] = u_f1[6, 6] = u_f1[7, 7] = 1

display(as_latex(u_f1, prefix=r"\mathbf{U_{f1}}="))
deutsch_jozsa(u_f1)

---
**Cell 02 - Constant 1 oracle.**
`u_f2` flips the ancilla for all four inputs ($f(x)=1$ everywhere).
Every row pair is swapped: the oracle is a block-diagonal matrix of
four $2\times 2$ swap blocks.
The algorithm detects this as constant.

In [ ]:
# Cell 02 - Constant 1 oracle: f(x)=1 for all x
# All four inputs flip the ancilla -> four swap blocks on diagonal

u_f2 = np.zeros((8, 8), dtype=complex)
u_f2[0, 1] = u_f2[1, 0] = u_f2[2, 3] = u_f2[3, 2] = 1
u_f2[4, 5] = u_f2[5, 4] = u_f2[6, 7] = u_f2[7, 6] = 1

display(as_latex(u_f2, prefix=r"\mathbf{U_{f2}}="))
deutsch_jozsa(u_f2)

---
**Cell 03 - Constant 0 oracle.**
`u_f3` leaves the ancilla unchanged for all four inputs ($f(x)=0$ everywhere).
The oracle is the $8\times 8$ identity matrix.
The algorithm detects this as constant.

In [ ]:
# Cell 03 - Constant 0 oracle: f(x)=0 for all x
# No input flips the ancilla -> identity matrix

u_f3 = np.zeros((8, 8), dtype=complex)
u_f3[0, 0] = u_f3[1, 1] = u_f3[2, 2] = u_f3[3, 3] = 1
u_f3[4, 4] = u_f3[5, 5] = u_f3[6, 6] = u_f3[7, 7] = 1

display(as_latex(u_f3, prefix=r"\mathbf{U_{f3}}="))
deutsch_jozsa(u_f3)

---
**Cell 04 - Balanced oracle (complement of u_f1).**
`u_f4` is the complement of `u_f1`: inputs $00$ and $01$ do not flip
the ancilla ($f=0$) while inputs $10$ and $11$ do ($f=1$).
Again exactly half the inputs give each output - balanced.
The two balanced oracles `u_f1` and `u_f4` illustrate that the
algorithm correctly identifies balanced functions regardless of
which half of the inputs produce $f=1$.

In [ ]:
# Cell 04 - Balanced oracle: f(00)=0, f(01)=0, f(10)=1, f(11)=1
# Rows 0-3: identity (no flip for inputs 00, 01)
# Rows 4-7: swap pairs (flip ancilla for inputs 10, 11)

u_f4 = np.zeros((8, 8), dtype=complex)
u_f4[0, 0] = u_f4[1, 1] = u_f4[2, 2] = u_f4[3, 3] = 1
u_f4[4, 5] = u_f4[5, 4] = u_f4[6, 7] = u_f4[7, 6] = 1

display(as_latex(u_f4, prefix=r"\mathbf{U_{f4}}="))
deutsch_jozsa(u_f4)